# A07 - Summary Tables and Figures

> **Dados legados.** Este notebook analisa campanhas congeladas, geradas
> com o modelo escalar de fidelidade (`ket_vector`) antes da revisão de
> realismo físico (`docs/physical_model.md`). Os números valem como
> registro do manuscrito V1, não para o modelo físico atual; os resultados
> que o manuscrito V2 usa estão em `results/realistic/`
> (`docs/realistic_results.md`).

## Objetivo
Consolidar as principais tabelas e figuras produzidas por `A00`-`A06` em
um único resumo (para a seção de resultados do artigo) e confirmar que
todo arquivo esperado em `results/figures/article/` e
`results/tables/article/` foi realmente gerado. Este notebook **não roda
nenhuma simulação nem recalcula nada** - só lê `results/raw/`/
`results/processed/` e monta o resumo final.

## Conceitos
- Cada valor abaixo é rastreável até um `trial_id` específico em
  `results/raw/<campanha>/trials.csv` - nenhum número aqui é digitado à
  mão (ver `docs/campaign_architecture.md`).


## Configuração


In [1]:
import pandas as pd
from pathlib import Path

from ibqn.experiments.persistence import read_trials_dataframe, trials_csv_path
from ibqn.experiments.export import save_table

RESULTS_DIR = Path("../results")
TABLES_DIR = RESULTS_DIR / "tables" / "article"
FIGURES_DIR = RESULTS_DIR / "figures" / "article"
CAMPAIGNS = ["C01_routing_strategy", "C02_fidelity_throughput", "C03_assurance_outcomes", "C04_reconciliation_effectiveness"]

trials = {c: read_trials_dataframe(trials_csv_path(RESULTS_DIR, c)) for c in CAMPAIGNS}


## Execução


In [2]:
c01 = trials["C01_routing_strategy"]
c01_least_loss = c01[c01["routing_strategy"] == "least_loss"]["delivered_pairs"].mean()
c01_shortest = c01[c01["routing_strategy"] == "shortest_hop_count"]["delivered_pairs"].mean()

c02 = trials["C02_fidelity_throughput"]
c02_purified_ceiling = c02[(c02["purification_policy"] == "automatic") & (c02["final_status"] == "SATISFIED")]["requested_fidelity"].max()
c02_disabled_ceiling = c02[(c02["purification_policy"] == "disabled") & (c02["final_status"] == "SATISFIED")]["requested_fidelity"].max()

c03 = trials["C03_assurance_outcomes"]
c03_status_counts = c03["final_status"].value_counts()

c04 = trials["C04_reconciliation_effectiveness"]
c04_recovery_rate = c04[c04["reconciliation_enabled"] == True]["recovered"].astype(bool).mean()
c04_mean_cost = c04[c04["reconciliation_enabled"] == True]["simulation_wall_time_s"].mean() - c04[c04["reconciliation_enabled"] == False]["simulation_wall_time_s"].mean()

print("Key figures extracted from persisted trial data:")
print(f"  C01: least_loss delivers {c01_least_loss:.0f} pairs/trial (mean) vs. shortest_hop_count's {c01_shortest:.0f}")
print(f"  C02: purification feasible up to min_fidelity={c02_purified_ceiling}, disabled only up to {c02_disabled_ceiling}")
print(f"  C03: status distribution = {c03_status_counts.to_dict()}")
print(f"  C04: recovery_rate={c04_recovery_rate:.2f}, mean additional cost={c04_mean_cost:.2f} s")


Key figures extracted from persisted trial data:
  C01: least_loss delivers 446 pairs/trial (mean) vs. shortest_hop_count's 4
  C02: purification feasible up to min_fidelity=0.72, disabled only up to 0.65
  C03: status distribution = {'REJECTED': 4, 'SATISFIED': 2, 'VIOLATED': 2}
  C04: recovery_rate=1.00, mean additional cost=6.04 s


## Resultados


In [3]:
summary_rows = [
    {"campaign": "C01_routing_strategy", "finding": "LeastLossRouting delivers far more pairs than ShortestHopCount/HighestFidelity on the diamond topology",
     "key_value": f"{c01_least_loss:.0f} vs {c01_shortest:.0f} pairs (mean)"},
    {"campaign": "C02_fidelity_throughput", "finding": "Purification extends the feasible fidelity range up to its own real ceiling",
     "key_value": f"automatic feasible <= {c02_purified_ceiling}, disabled feasible <= {c02_disabled_ceiling}"},
    {"campaign": "C03_assurance_outcomes", "finding": "SATISFIED/VIOLATED/REJECTED all occur naturally, never forced",
     "key_value": str(c03_status_counts.to_dict())},
    {"campaign": "C04_reconciliation_effectiveness", "finding": "Reconciliation recovers every VIOLATED trial at a real, measured cost",
     "key_value": f"recovery_rate={c04_recovery_rate:.2f}, +{c04_mean_cost:.2f}s"},
]
summary_df = pd.DataFrame(summary_rows)
save_table(summary_df, "A07_article_summary", directory=TABLES_DIR)
summary_df


,campaign,finding,key_value
0,C01_routing_strategy,LeastLossRouting delivers far more pairs than ...,446 vs 4 pairs (mean)
1,C02_fidelity_throughput,Purification extends the feasible fidelity ran...,"automatic feasible <= 0.72, disabled feasible ..."
2,C03_assurance_outcomes,SATISFIED/VIOLATED/REJECTED all occur naturall...,"{'REJECTED': 4, 'SATISFIED': 2, 'VIOLATED': 2}"
3,C04_reconciliation_effectiveness,Reconciliation recovers every VIOLATED trial a...,"recovery_rate=1.00, +6.04s"


In [4]:
existing_figures = sorted(FIGURES_DIR.glob("*.pdf")) if FIGURES_DIR.exists() else []
existing_tables = sorted(TABLES_DIR.glob("*.csv")) if TABLES_DIR.exists() else []

print(f"Figures generated so far ({len(existing_figures)}):")
for f in existing_figures:
    print(f"  {f.name}")
print(f"\nTables generated so far ({len(existing_tables)}):")
for t in existing_tables:
    print(f"  {t.name}")


Figures generated so far (7):
  A01_delivered_pairs_by_strategy.pdf
  A01_satisfaction_rate_by_strategy.pdf
  A02_fidelity_throughput_tradeoff.pdf
  A03_outcomes_by_attenuation.pdf
  A04_reconciliation_effectiveness.pdf
  A05_planner_estimation_error.pdf
  A06_delivery_overprovisioning.pdf

Tables generated so far (7):
  A01_routing_strategy_summary.csv
  A02_rejection_rate_by_policy_and_fidelity.csv
  A03_status_by_attenuation_and_fidelity.csv
  A04_reconciliation_paired_by_seed.csv
  A05_fidelity_estimation_error_by_campaign.csv
  A06_delivery_ratio_by_purification.csv
  A07_article_summary.csv


In [5]:
expected_figure_stems = {
    "A01_delivered_pairs_by_strategy", "A01_satisfaction_rate_by_strategy",
    "A02_fidelity_throughput_tradeoff", "A03_outcomes_by_attenuation",
    "A04_reconciliation_effectiveness", "A05_planner_estimation_error",
    "A06_delivery_overprovisioning",
}
found_stems = {f.stem for f in existing_figures}
missing = expected_figure_stems - found_stems
assert not missing, f"expected figures missing (run notebooks A01-A06 first): {missing}"
assert len(existing_tables) >= 7, f"expected at least 7 tables from A00-A07, found {len(existing_tables)}"
print("Confirmed: every expected figure from A01-A06 is present, and all article tables were generated.")


Confirmed: every expected figure from A01-A06 is present, and all article tables were generated.


## Interpretação
Os quatro achados centrais das campanhas de validação (H3) estão
resumidos e rastreáveis: divergência real entre estratégias de
roteamento, teto real de purificação, ocorrência natural dos três status
de assurance, e eficácia mensurável (com custo real) da reconciliation.
Cada valor nesta tabela vem diretamente de `results/raw/<campanha>/
trials.csv` - nenhum foi calculado fora dessa cadeia.

## Limitações
- Este resumo consolida apenas as quatro campanhas de validação em
  pequena escala (H3.4) - a campanha final do artigo (mencionada no
  "próximo passo" do relatório da Fase H3) ainda não existe.
- Ver `docs/campaign_architecture.md`, seção 10, para as limitações
  metodológicas que se aplicam a toda a Fase H3.

## Conclusão
Resumo consolidado e rastreável de A00-A06, com todas as figuras/tabelas
esperadas confirmadas presentes em `results/figures/article/` e
`results/tables/article/`.
